# `app:robustness` table for the paper

The controlled uncertainty-error correlation **per variation, per cohort and per head, with the per-donor interval** -- `app:robustness` of the paper.

The cascade is the one of `unc_error_correlations_classic.ipynb` and nothing about it changes here: `n >= 60` group filter, group-size-weighted fold to (gene, donor) on the gene axis, plain mean over spots on the spot axis, then `donor_lopo_stats` over the K held-out donors. **Only the SOURCE of the parquet changes from one variation to the next.** That is the whole design: a variation must move exactly one knob and nothing else, otherwise the table compares two things at once.

Where each variation comes from:

| variation | run_id | canal2 dir | error case |
|---|---|---|---|
| reference (the paper's `fig:axes`) | `pr_classic3` | `canal2_uncertainty` | `lin_all` |
| detected entries only | `pr_classic3` | `canal2_uncertainty` | `lin_pos` |
| absolute log-ratio error | `pr_classic3` | `canal2_uncertainty` | `log_pos` |
| epoch selection on validation NLL | `pr_classic4_vloss_v1` | `canal2_uncertainty` | `lin_all` |
| epoch selection on MSE | `pr_classic7_mse_sel` | `canal2_uncertainty` | `lin_all` |
| beta-NLL loss | `pr_classic3`, `lambda_{beta}` | `canal2_uncertainty` | `lin_all` |
| count heads on raw counts | `pr_classic3`, `*_raw` heads | `canal2_uncertainty` | `lin_all` |
| uncertainty = coefficient of variation | `pr_classic3` | `canal2_uncertainty_cv` | `lin_all` |
| uncertainty = 1 + alpha | `pr_classic3` | `canal2_uncertainty_one_plus_alpha` | `lin_all` |

The first three read the SAME directory and differ only by which column of the parquet is read, so they cost one read for three rows. The last two need `canal2_uncertainty_precompute.py --uncertainty-form`, launched by `snake/scripts/canals/launch_canal2_uform_precompute.sh`.

Output: the printed table (the authority), plus `table_robustness_paper.tex` written next to the paper figures.

In [7]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))   # parent dir of src (notebooks are not pip-installed on the cluster)
from src.evaluation.benchmark_summary import donor_lopo_stats

# =============================== SETTINGS ====================================
RESULTS  = PROJ / "results"
DATASETS = ["10x_TuPro", "TLS", "LUNG_CANCER"]
RADIUS   = "r75"
OUTDIR   = PROJ / "results" / "_analysis" / "figures_paper"

# TWO ways to be MUTE, and only one justifies dropping the row.
#   (a) Mute on BOTH statistics -- no marker in fig:axes. The standard deviation is CONSTANT
#       within the group, so NO correlation exists: gene-homoscedastic on the GENE axis,
#       spot-homoscedastic on the SPOT axis. The row would be '-- --' in all three cohorts and
#       says nothing the head's parametrization does not already say. DROPPED.
#   (b) Mute on the CONTROLLED one only -- hollow marker, no filled one. The standard deviation
#       varies, but as a monotone function of the mean: the Poisson. The UNcontrolled value
#       exists and IS the result, so the row STAYS.
# The drop NEVER happens when an artifact is missing (r is None): that case must stay
# visible as a '?', not disappear. Set False to keep everything.
DROP_FULLY_SILENT_ROWS = True
# =============================================================================

N_PLUS_MIN = 60          # the script's inclusion filter, RE-APPLIED here (the parquets carry
                         # EVERY group present: filtering is the reader's job)
CONVENTION = "v3_native_scale"
AXES = ["gene", "spot"]
# The group size depends on the case's POPULATION, not only on the axis.
N_COL = {"lin_all": "n_plus_all", "lin_pos": "n_plus_pos", "log_pos": "n_plus_pos"}

GAUSS  = ["homoscedastic", "heteroscedastic", "spot_homoscedastic"]
COUNTS = ["poisson_cp", "negativebinomial_hetero_noanchor_cp"]
HEADS5 = GAUSS + COUNTS                 # the five heads of fig:axes
RAW    = ["poisson_raw", "negativebinomial_hetero_noanchor_raw"]
NB     = ["negativebinomial_hetero_noanchor_cp"]
# beta-NLL: the THREE Gaussians. The gene-homoscedastic one is proven beta-invariant (log_var
# constant over the batch, self-test in losses.py), so its row must REPRODUCE the reference --
# a free internal control, not a redundancy.
GAUSS_BETA = GAUSS

MODEL_LABELS = {
    "homoscedastic":                        "Gaussian gene-homoscedastic",
    "heteroscedastic":                      "Gaussian heteroscedastic",
    "spot_homoscedastic":                   "Gaussian spot-homoscedastic",
    "poisson_cp":                           "Poisson",
    "negativebinomial_hetero_noanchor_cp":  "Negative binomial",
    "poisson_raw":                          "Poisson (raw counts)",
    "negativebinomial_hetero_noanchor_raw": "Negative binomial (raw counts)",
}
DATASET_LABELS = {"10x_TuPro": "Tumor Profiler", "TLS": "Kidney/Lung TLS",
                  "LUNG_CANCER": "NSCLC"}

# ── The variations ───────────────────────────────────────────────────────────────────────────
# ONE entry = one set of parquets + ONE error case + a list of heads. `lam` is the lambda_ value
# to keep for the Gaussian heads (the count heads have no such directory level and ignore it).
# lambda_0.0 IS the true NLL, hence the reference.
def V(key, label, models, run_id="pr_classic3", canal2="canal2_uncertainty", case="lin_all",
      lam="0.0"):
    return dict(key=key, label=label, models=models, run_id=run_id, canal2=canal2, case=case,
                lam=lam)

VARIATIONS = [
    V("ref",        "Reference (fig:axes)",                    HEADS5),
    V("detected",   "Detected entries only",                   HEADS5, case="lin_pos"),
    V("logratio",   "Absolute log-ratio error",                HEADS5, case="log_pos"),
    V("sel_vloss",  "Epoch selection: validation NLL",         HEADS5, run_id="pr_classic4_vloss_v1"),
    V("sel_mse",    "Epoch selection: MSE",                    HEADS5, run_id="pr_classic7_mse_sel"),
    V("beta025",    "beta-NLL, beta = 0.25",                   GAUSS_BETA, lam="0.25"),
    V("beta050",    "beta-NLL, beta = 0.5",                    GAUSS_BETA, lam="0.5"),
    V("beta075",    "beta-NLL, beta = 0.75",                   GAUSS_BETA, lam="0.75"),
    V("beta100",    "beta-NLL, beta = 1",                      GAUSS_BETA, lam="1.0"),
    V("raw",        "Count heads trained on raw counts",       RAW),
    V("cv",         "Uncertainty = coefficient of variation",  HEADS5, canal2="canal2_uncertainty_cv"),
    V("alpha",      "Uncertainty = 1 + alpha",                 NB,
      canal2="canal2_uncertainty_one_plus_alpha"),
]

# -- slide -> donor -----------------------------------------------------------------------
# SPLIT unit = donor, METRIC unit = slide (src/data/splitting.py).
DONOR_MAP = yaml.safe_load((PROJ / "snake" / "config" / "donor_slide_map.yaml").read_text())

def slide_to_donor(ds):
    m = DONOR_MAP.get(ds) or {}
    return {sl: d for d, slides in m.items() for sl in slides}


def canal2_dir(ds, v, model):
    """The canal2 directory of one (cohort, variation, head), or None if it does not exist.

    Two possible locations for one head: the Gaussians carry a lambda_ level, the count heads
    do not. So the lambda_ level is tried FIRST, then the bare directory, rather than branching
    on a list of families that would have to be kept up to date."""
    base = (RESULTS / ds / v["run_id"] / f"_{ds}_hvg_top3000"
            / RADIUS / model)
    for leaf in ([base / f"lambda_{v['lam']}"] if v["lam"] else []) + [base]:
        d = leaf / v["canal2"]
        if (d / "summary.json").is_file():
            return d
    return None


print(f"{len(VARIATIONS)} variations x {len(DATASETS)} cohorts")
for v in VARIATIONS:
    print(f"  {v['key']:<10s} {v['label']:<40s} {v['run_id']:<22s} {v['canal2']:<34s} "
          f"{v['case']:<8s} lam={v['lam']}  {len(v['models'])} head(s)")

12 variations x 3 cohorts
  ref        Reference (fig:axes)                     pr_classic3            canal2_uncertainty                 lin_all  lam=0.0  5 head(s)
  detected   Detected entries only                    pr_classic3            canal2_uncertainty                 lin_pos  lam=0.0  5 head(s)
  logratio   Absolute log-ratio error                 pr_classic3            canal2_uncertainty                 log_pos  lam=0.0  5 head(s)
  sel_vloss  Epoch selection: validation NLL          pr_classic4_vloss_v1   canal2_uncertainty                 lin_all  lam=0.0  5 head(s)
  sel_mse    Epoch selection: MSE                     pr_classic7_mse_sel    canal2_uncertainty                 lin_all  lam=0.0  5 head(s)
  beta025    beta-NLL, beta = 0.25                    pr_classic3            canal2_uncertainty                 lin_all  lam=0.25  3 head(s)
  beta050    beta-NLL, beta = 0.5                     pr_classic3            canal2_uncertainty                 lin_all  lam=0.5  3 h

## 1. The donor cascade, and the printed table

Identical to `unc_error_correlations_classic.ipynb`: the `n >= 60` filter, the group-size-weighted fold and the donor boundary are the script's. On the gene axis the fold is in two steps, groups to `(gene, donor)` weighted by size, then a simple mean over the donor's genes. On the spot axis a spot belongs to one slide hence one donor, and the mean is direct.

Two ways of being mute, both by construction and neither a missing value: no marker at all means the predicted spread is constant inside the group, and a naive value with no controlled one means the spread varies but is a monotone function of the predicted mean.

In [8]:
def donor_points(frame, ds, axis, stat, case):
    """{donor: value} + the number of groups kept, for ONE (axis, statistic, case)."""
    col, ncol = f"{stat}_{case}", N_COL[case]
    if frame is None or col not in frame.columns or ncol not in frame.columns:
        return {}, 0
    d = frame[(frame[ncol] >= N_PLUS_MIN) & frame[col].notna()]
    if d.empty:
        return {}, 0
    s2d = slide_to_donor(ds)
    donor = d["slide"].map(lambda s: s2d.get(s, s))
    if axis == "spot":                       # one spot = one slide = one donor: plain mean
        return d.groupby(donor.to_numpy())[col].mean().to_dict(), int(len(d))
    g = pd.DataFrame({"gene": d["gene"].to_numpy(), "donor": donor.to_numpy(),
                      "w": d[col].to_numpy() * d[ncol].to_numpy(),
                      "n": d[ncol].to_numpy()}).groupby(["gene", "donor"], as_index=False).sum()
    g["v"] = g["w"] / g["n"]                 # pair -> (gene, donor), weighted by n
    return g.groupby("donor")["v"].mean().to_dict(), int(len(d))


# The first three variations read the SAME directory and differ only by the column:
# the cache avoids re-reading the same parquets three times (the biggest files of the lot).
_CACHE = {}
def frames(d):
    if d not in _CACHE:
        def rd(name):
            p = d / name
            return pd.read_parquet(p) if p.is_file() else None
        _CACHE[d] = (rd("pair_partial.parquet"), rd("spot_partial.parquet"))
    return _CACHE[d]


rows, missing, stale, notapp = [], [], [], []
for v in VARIATIONS:
    for ds in DATASETS:
        for m in v["models"]:
            d = canal2_dir(ds, v, m)
            if d is None:
                missing.append((v["key"], ds, m))
                continue
            s = json.loads((d / "summary.json").read_text())
            # GUARD: an artifact on the other column convention carries a converted scale.
            # Mixing it in silently is the one way to be wrong without noticing, so it is
            # dropped by name.
            if s.get("canal2_convention") != CONVENTION:
                stale.append((v["key"], ds, m, s.get("canal2_convention")))
                continue
            if s.get("not_applicable"):
                notapp.append((v["key"], ds, m, s.get("reason")))
                continue
            pair, spot = frames(d)
            for axis in AXES:
                rec = dict(variation=v["key"], label=v["label"], dataset=ds, model=m, axis=axis,
                           case=v["case"],
                           # cv leaves the uncertainty undefined where mu <= 0: the population is
                           # then not the reference's any more, and this says by how much.
                           pct_dropped=float(s.get("uncertainty_undefined_pct_entries_dropped", 0.0)))
                for stat, tag in (("r_uD", "naive"), ("r_partial", "partial")):
                    pts, n = donor_points(pair if axis == "gene" else spot, ds, axis, stat,
                                          v["case"])
                    if not pts:
                        continue
                    st = donor_lopo_stats(pts)
                    rec.update({tag: st["mean"], f"{tag}_lo": st["ci95_mean"][0],
                                f"{tag}_hi": st["ci95_mean"][1], "K": st["K"], "n_groups": n})
                rows.append(rec)
RES = pd.DataFrame(rows)
assert not RES.empty, (f"no canal2 artifact found at all under {RESULTS} -- check the run_ids and "
                       f"that the sweep ran; {len(missing)} (variation, cohort, head) triples had "
                       f"no summary.json, e.g. {missing[:3]}")
for c in ("naive", "naive_lo", "naive_hi", "partial", "partial_lo", "partial_hi", "K", "n_groups"):
    if c not in RES.columns:
        RES[c] = np.nan


def get(vkey, ds, m, axis):
    r = RES[(RES["variation"] == vkey) & (RES["dataset"] == ds) & (RES["model"] == m)
            & (RES["axis"] == axis)]
    return None if r.empty else r.iloc[0]


def cell(r, tag, width=25):
    """One cell: the value, its Student CI over the donors, and '*' if the CI excludes zero."""
    if r is None or not np.isfinite(r.get(tag, np.nan)):
        return f"{'mute':>{width}s}"
    lo, hi = r.get(f"{tag}_lo", np.nan), r.get(f"{tag}_hi", np.nan)
    if not np.isfinite(lo):
        return f"{r[tag]:>+{width}.4f}"
    star = "*" if (lo > 0 or hi < 0) else " "
    return f"{r[tag]:+.3f} [{lo:+.3f},{hi:+.3f}]{star}".rjust(width)


print("#" * 150)
print("# ROBUSTNESS CHECKS -- uncertainty-error Spearman correlation, uncontrolled and controlled")
print("# for the predicted mean. Interval: Student-t 95% over the held-out donors. '*' = excludes 0.")
print("#" * 150)
for axis in AXES:
    grouping = ("across the SPOTS of one gene" if axis == "gene"
                else "across the GENES of one spot")
    print()
    print("=" * 150)
    print(f"AXIS {axis.upper()} -- one group per {'(gene, slide)' if axis == 'gene' else '(slide, spot)'}, "
          f"{grouping}")
    print("=" * 150)
    for v in VARIATIONS:
        print()
        print(f"--- {v['label']}   [{v['run_id']} | {v['canal2']} | {v['case']}"
              f"{'' if not v['lam'] else ' | lambda_' + v['lam']}]")
        print(f"  {'head':<32s} " + " ".join(f"{DATASET_LABELS[ds] + ' raw':>25s} "
                                             f"{DATASET_LABELS[ds] + ' ctrl':>25s}"
                                             for ds in DATASETS))
        for m in v["models"]:
            cells = []
            for ds in DATASETS:
                r = get(v["key"], ds, m, axis)
                cells += [cell(r, "naive"), cell(r, "partial")]
            print(f"  {MODEL_LABELS.get(m, m):<32s} " + " ".join(cells))

print()
print("=" * 150)
print("PROVENANCE")
print("=" * 150)
for ds in DATASETS:
    sub = RES[RES["dataset"] == ds]
    k = int(sub["K"].max()) if sub["K"].notna().any() else 0
    print(f"  {DATASET_LABELS[ds]:<18s} K = {k} donors")
cv_rows = RES[(RES["variation"] == "cv") & (RES["axis"] == "gene")]
if not cv_rows.empty:
    print()
    print("  cv = sd/mu is UNDEFINED where mu <= 0; those entries are dropped, so its population is")
    print("  not exactly the reference one. Share of positive entries dropped, per run:")
    for _, r in cv_rows.iterrows():
        print(f"    {DATASET_LABELS[r['dataset']]:<18s} {MODEL_LABELS.get(r['model'], r['model']):<32s} "
              f"{r['pct_dropped']:.3f}%")
for name, lst in (("MISSING (no summary.json -- not computed yet?)", missing),
                  ("NOT APPLICABLE (sentinel written by the script)", notapp),
                  (f"STALE (canal2_convention != {CONVENTION!r}) -- DISCARDED", stale)):
    if lst:
        print()
        print(f"  (!) {len(lst)} {name}:")
        for t in lst:
            print("      " + "  ".join(str(x) for x in t))

######################################################################################################################################################
# ROBUSTNESS CHECKS -- uncertainty-error Spearman correlation, uncontrolled and controlled
# for the predicted mean. Interval: Student-t 95% over the held-out donors. '*' = excludes 0.
######################################################################################################################################################

AXIS GENE -- one group per (gene, slide), across the SPOTS of one gene

--- Reference (fig:axes)   [pr_classic3 | canal2_uncertainty | lin_all | lambda_0.0]
  head                                    Tumor Profiler raw       Tumor Profiler ctrl       Kidney/Lung TLS raw      Kidney/Lung TLS ctrl                 NSCLC raw                NSCLC ctrl
  Gaussian gene-homoscedastic                           mute                      mute                      mute                      mute                      mute

## 2. The LaTeX table

The cell below prints **the LaTeX and nothing else**. Two `table` environments, one per grouping, in the house style of `tab:estimator-gap` in that file: caption first, no bold lead sentence, booktabs, `$\pm$` set tight.

Rows are (variation, head), columns are the three cohorts, each holding the uncontrolled value and the controlled one with the half-width of its Student-t 95% interval over the K held-out donors. The explicit `[lo, hi]` bounds are in the printed table of section 1, which stays the authority.



In [9]:
# LaTeX names. Kept apart from the PRINTED table's: that one stays ASCII, readable in a
# terminal, this one carries the math and must fit a narrow column. The "Gaussian" qualifier
# goes into the caption rather than into every row: it costs a whole column.
#
# ONE LIST PER VARIATION, AND THAT IS THE POINT: each element is ONE ROW of the block. A label
# that is too long is NOT put in a fixed-width p{...} column -- that would fill the right of
# every short label with white space and inflate the column for nothing. It is SPREAD over the
# rows the block ALREADY has (one per head), so the column is only as wide as the longest
# FRAGMENT, and the table gains neither a row nor a blank. The break is written by hand rather
# than computed: twelve labels, and an automatic break would not know where to cut a
# formula.
TEX_VAR_LABELS = {
    "ref":       ["Reference"],
    "detected":  ["Detected", "entries only"],
    "logratio":  ["Absolute", "log-ratio error"],
    "sel_vloss": ["Selection:", "validation NLL"],
    "sel_mse":   ["Selection: MSE"],
    "beta025":   [r"$\beta$-NLL, $\beta = 0.25$"],
    "beta050":   [r"$\beta$-NLL, $\beta = 0.5$"],
    "beta075":   [r"$\beta$-NLL, $\beta = 0.75$"],
    "beta100":   [r"$\beta$-NLL, $\beta = 1$"],
    "raw":       ["Raw-count", "target"],
    # sigma is reserved for the Gaussian (cf. app:metrics: sigma for the Gaussians, sqrt(mu)
    # for the Poisson, sqrt(mu + alpha mu^2) for the NB). These two rows cover several
    # families, so they name the SQUARE ROOT OF THE VARIANCE, not sigma. The word "Uncertainty"
    # gives way to "u =", defined in the caption: the 1+alpha block has only ONE row
    # (the NB), so its label cannot be spread and must stand alone.
    "cv":        [r"$u = \sqrt{\mathrm{Var}}/\hat{\mu}$"],
    "alpha":     [r"$u = 1 + \alpha$"],
}
TEX_MODEL_LABELS = {
    "homoscedastic":                        "Gene-homoscedastic",
    "heteroscedastic":                      "Heteroscedastic",
    "spot_homoscedastic":                   "Spot-homoscedastic",
    "poisson_cp":                           "Poisson",
    "negativebinomial_hetero_noanchor_cp":  "Negative binomial",
    "poisson_raw":                          "Poisson",
    "negativebinomial_hetero_noanchor_raw": "Negative binomial",
}

# K per cohort, read from the data and not hardcoded: it is the number of points that carry
# the interval. It lives in the CAPTION and not in the header, where "($K=7$)" widened every
# cohort column without telling the reader anything not said once and for all.
K_BY_DS = {ds: (int(RES[RES["dataset"] == ds]["K"].max())
                if RES[RES["dataset"] == ds]["K"].notna().any() else 0) for ds in DATASETS}

# TWO absences, and they do not mean the same thing, so they do not carry the same symbol.
# A dash = the statistic does NOT EXIST (head mute by construction). That is a result, and it
# goes into the paper. A '?' = the canal2 artifact was not read (missing, or dropped as not
# conforming to the convention), so the cell is not a result. It is marked loudly rather than
# left to look like one. The exact list is printed by the previous cell, under MISSING / STALE.
MISSING_MARK = r"\textbf{?}"


def tex_num(r, tag, with_ci):
    """One cell. Half-width of the Student CI over the donors, not the two bounds: it is
    symmetric by construction, and the explicit bounds are in the printed table above.
    Tight $\\pm$ format, as in tab:estimator-gap."""
    if r is None:
        return MISSING_MARK
    if not np.isfinite(r.get(tag, np.nan)):
        return "--"
    v = r[tag]
    lo, hi = r.get(f"{tag}_lo", np.nan), r.get(f"{tag}_hi", np.nan)
    if not with_ci or not np.isfinite(lo):
        return f"${v:+.2f}$"
    return f"${v:+.2f}" + r"\pm" + f"{(hi - lo) / 2:.2f}$"


def row_is_fully_silent(vkey, m, axis):
    """The head has NEITHER statistic, in ANY cohort, on this axis.

    This is case (a) of the config cell: the standard deviation is constant within the group, so
    neither the naive nor the controlled one exists (gene-homoscedastic on the GENE axis,
    spot-homoscedastic on the SPOT axis). A MISSING artifact (r is None) does not count as
    silence -- it must stay visible as '?' -- hence the `all(r is not None ...)`. Defined here
    and reused by the render cell, which must draw exactly the same rows."""
    recs = [get(vkey, ds, m, axis) for ds in DATASETS]
    if not all(r is not None for r in recs):
        return False
    return all(not np.isfinite(r.get("naive", np.nan))
               and not np.isfinite(r.get("partial", np.nan)) for r in recs)


def kept_heads(v, axis):
    return [m for m in v["models"]
            if not (DROP_FULLY_SILENT_ROWS and row_is_fully_silent(v["key"], m, axis))]


def var_lines(key, n_rows):
    """The variation's label, one cell per block row, padded with empty cells.

    If the block has FEWER rows than the label has fragments, the last available row absorbs
    the rest: it then exceeds the intended width, which shows immediately and
    stays preferable to a piece of label lost in silence."""
    lines = list(TEX_VAR_LABELS[key])
    if n_rows == 0:
        return []
    if len(lines) > n_rows:
        lines = lines[:n_rows - 1] + [" ".join(lines[n_rows - 1:])]
    return lines + [""] * (n_rows - len(lines))


def caption(axis):
    grouping = ("across the spots of each gene" if axis == "gene"
                else "across the genes of each spot")
    silent = ("the gene-homoscedastic Gaussian" if axis == "gene"
              else "the spot-homoscedastic Gaussian")
    donors = ", ".join(f"${K_BY_DS[ds]}$" for ds in DATASETS[:-1]) + f" and ${K_BY_DS[DATASETS[-1]]}$"
    return ("Spearman correlation between predicted uncertainty and absolute error under each "
            "modeling and scoring variation, before and after controlling for the predicted mean. "
            f"Correlations are computed {grouping}, then averaged over the held-out donors of the "
            f"cohort ({donors} donors, left to right); controlled values carry the half-width of "
            r"the donor Student-$t$ $95\%$ interval. Rows use the reference setting except where "
            r"stated: absolute error over all entries, the predictive standard deviation as the "
            r"uncertainty $u$, and the epoch selected on validation correlation. The three "
            "Gaussian heads are named by their variance structure. A dash marks a correlation "
            "that does not exist on this grouping, the predicted spread being a monotone function "
            f"of the predicted mean. {silent.capitalize()} is omitted entirely, its predicted "
            "spread being constant within the group so that neither correlation is defined.")


def emit_table(axis):
    L = []
    L.append(r"\begin{table}[t]")
    L.append(r"\caption{" + caption(axis) + "}")
    L.append(r"\label{tab:robustness-" + axis + "}")
    L.append(r"\centering")
    L.append(r"\scriptsize")
    L.append(r"\setlength{\tabcolsep}{3pt}")
    L.append(r"\begin{tabular}{llrrrrrr}")
    L.append(r"\toprule")
    L.append(" & ".join(["", ""] + [r"\multicolumn{2}{c}{" + DATASET_LABELS[ds] + "}"
                                    for ds in DATASETS]) + r" \\")
    L.append(r"\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}")
    L.append("Variation & Head & "
             + " & ".join(["uncontrolled & controlled"] * len(DATASETS)) + r" \\")
    L.append(r"\midrule")
    emitted = 0
    for v in VARIATIONS:
        # The KEPT heads first: the label is spread over them, so they must be counted
        # before writing the block's first row.
        kept = kept_heads(v, axis)
        if not kept:
            continue
        if emitted:
            L.append(r"\addlinespace")
        for lab, m in zip(var_lines(v["key"], len(kept)), kept):
            cells = []
            for ds in DATASETS:
                r = get(v["key"], ds, m, axis)
                cells += [tex_num(r, "naive", False), tex_num(r, "partial", True)]
            L.append(lab + " & " + TEX_MODEL_LABELS.get(m, m) + " & " + " & ".join(cells) + r" \\")
            emitted += 1
    L.append(r"\bottomrule")
    L.append(r"\end{tabular}")
    L.append(r"\end{table}")
    return "\n".join(L)


tex = "\n\n".join(emit_table(axis) for axis in AXES)
OUTDIR.mkdir(parents=True, exist_ok=True)
(OUTDIR / "table_robustness_paper.tex").write_text(tex)   # silent copy to disk
print(tex)

\begin{table}[t]
\caption{Spearman correlation between predicted uncertainty and absolute error under each modeling and scoring variation, before and after controlling for the predicted mean. Correlations are computed across the spots of each gene, then averaged over the held-out donors of the cohort ($7$, $6$ and $8$ donors, left to right); controlled values carry the half-width of the donor Student-$t$ $95\%$ interval. Rows use the reference setting except where stated: absolute error over all entries, the predictive standard deviation as the uncertainty $u$, and the epoch selected on validation correlation. The three Gaussian heads are named by their variance structure. A dash marks a correlation that does not exist on this grouping, the predicted spread being a monotone function of the predicted mean. The gene-homoscedastic gaussian is omitted entirely, its predicted spread being constant within the group so that neither correlation is defined.}
\label{tab:robustness-gene}
\centeri

## 3. Rendered preview

The same two tables as rendered frames, for a quick visual check. Not a deliverable: the LaTeX of section 2 is what goes in the paper, and the printed table of section 1 stays the authority for the exact numbers. This view reads `RES` through the same `get()` as the LaTeX emitter, so it cannot disagree with it.

`--` means the correlation does not exist on that grouping (mute by construction), `?` means the canal2 artifact was not read (see MISSING / STALE in section 1).

In [10]:
from IPython.display import display

# The RENDERED TWIN of the LaTeX above: same rows (via kept_heads, defined by the LaTeX cell),
# same columns, same rounding. It recomputes NOTHING (it re-reads RES via get()), so it
# cannot diverge from the table that goes into the paper. Only the formatting changes: '±' in
# place of $\pm$, and the ASCII labels of the config cell instead of the LaTeX ones.
def txt_num(r, tag, with_ci):
    if r is None:
        return "?"                            # artifact not read -- see MISSING / STALE above
    if not np.isfinite(r.get(tag, np.nan)):
        return "--"                           # mute by construction
    v = r[tag]
    lo, hi = r.get(f"{tag}_lo", np.nan), r.get(f"{tag}_hi", np.nan)
    if not with_ci or not np.isfinite(lo):
        return f"{v:+.2f}"
    return f"{v:+.2f} ± {(hi - lo) / 2:.2f}"


def render_table(axis):
    idx, data = [], []
    for v in VARIATIONS:
        for m in kept_heads(v, axis):
            idx.append((v["label"], MODEL_LABELS.get(m, m)))
            row = []
            for ds in DATASETS:
                r = get(v["key"], ds, m, axis)
                row += [txt_num(r, "naive", False), txt_num(r, "partial", True)]
            data.append(row)
    cols = pd.MultiIndex.from_product(
        [[f"{DATASET_LABELS[ds]} (K={K_BY_DS[ds]})" for ds in DATASETS],
         ["uncontrolled", "controlled"]])
    return pd.DataFrame(data, columns=cols,
                        index=pd.MultiIndex.from_tuples(idx, names=["Variation", "Head"]))


pd.set_option("display.max_rows", None)       # nothing must be truncated
for axis in AXES:
    grouping = ("across the spots of each gene" if axis == "gene"
                else "across the genes of each spot")
    dropped = sorted({MODEL_LABELS.get(m, m) for v in VARIATIONS for m in v["models"]
                      if m not in kept_heads(v, axis)})
    print(f"\nAXIS {axis.upper()} -- {grouping}   (Spearman, controlled = mean partialled out; "
          f"± = half-width of the donor Student-t 95% interval)")
    if dropped:
        print(f"  omitted, no correlation defined on this grouping: {', '.join(dropped)}")
    display(render_table(axis))


AXIS GENE -- across the spots of each gene   (Spearman, controlled = mean partialled out; ± = half-width of the donor Student-t 95% interval)
  omitted, no correlation defined on this grouping: Gaussian gene-homoscedastic


Tumor Profiler (K=7)  \
                                                                              uncontrolled   
Variation                              Head                                                  
Reference (fig:axes)                   Gaussian heteroscedastic                      +0.20   
                                       Gaussian spot-homoscedastic                   +0.06   
                                       Poisson                                       +0.49   
                                       Negative binomial                             +0.36   
Detected entries only                  Gaussian heteroscedastic                      +0.11   
                                       Gaussian spot-homoscedastic                   +0.15   
                                       Poisson                                       +0.00   
                                       Negative binomial                             +0.06   
Absolute log-ratio error               Gaussian heteroscedastic                      -0.05   
                                       Gaussian spot-homoscedastic                   +0.08   
                                       Poisson                                       -0.26   
                                       Negative binomial                             -0.15   
Epoch selection: validation NLL        Gaussian heteroscedastic                      +0.13   
                                       Gaussian spot-homoscedastic                   +0.08   
                                       Poisson                                       +0.49   
                                       Negative binomial                             +0.42   
Epoch selection: MSE                   Gaussian heteroscedastic                      +0.15   
                                       Gaussian spot-homoscedastic                   +0.06   
                                       Poisson                                       +0.49   
                                       Negative binomial                             +0.37   
beta-NLL, beta = 0.25                  Gaussian heteroscedastic                      +0.19   
                                       Gaussian spot-homoscedastic                   +0.05   
beta-NLL, beta = 0.5                   Gaussian heteroscedastic                      +0.18   
                                       Gaussian spot-homoscedastic                   +0.05   
beta-NLL, beta = 0.75                  Gaussian heteroscedastic                      +0.17   
                                       Gaussian spot-homoscedastic                   +0.05   
beta-NLL, beta = 1                     Gaussian heteroscedastic                      +0.15   
                                       Gaussian spot-homoscedastic                   +0.04   
Count heads trained on raw counts      Poisson (raw counts)                          +0.54   
                                       Negative binomial (raw counts)                +0.54   
Uncertainty = coefficient of variation Gaussian gene-homoscedastic                   -0.40   
                                       Gaussian heteroscedastic                      -0.28   
                                       Gaussian spot-homoscedastic                   -0.35   
                                       Poisson                                       -0.49   
                                       Negative binomial                             -0.23   
Uncertainty = 1 + alpha                Negative binomial                             -0.03   

                                                                                     \
                                                                         controlled   
Variation                              Head                                           
Reference (fig:axes)                   Gaussian heteroscedastic        +0.06 ± 0.02   
                                       Gaussian spot-ho


AXIS SPOT -- across the genes of each spot   (Spearman, controlled = mean partialled out; ± = half-width of the donor Student-t 95% interval)
  omitted, no correlation defined on this grouping: Gaussian spot-homoscedastic


Tumor Profiler (K=7)  \
                                                                              uncontrolled   
Variation                              Head                                                  
Reference (fig:axes)                   Gaussian gene-homoscedastic                   +0.34   
                                       Gaussian heteroscedastic                      +0.66   
                                       Poisson                                       +0.79   
                                       Negative binomial                             +0.79   
Detected entries only                  Gaussian gene-homoscedastic                   -0.09   
                                       Gaussian heteroscedastic                      -0.02   
                                       Poisson                                       +0.34   
                                       Negative binomial                             +0.34   
Absolute log-ratio error               Gaussian gene-homoscedastic                   -0.35   
                                       Gaussian heteroscedastic                      -0.43   
                                       Poisson                                       -0.36   
                                       Negative binomial                             -0.34   
Epoch selection: validation NLL        Gaussian gene-homoscedastic                   +0.60   
                                       Gaussian heteroscedastic                      +0.64   
                                       Poisson                                       +0.78   
                                       Negative binomial                             +0.75   
Epoch selection: MSE                   Gaussian gene-homoscedastic                   -0.20   
                                       Gaussian heteroscedastic                      +0.66   
                                       Poisson                                       +0.79   
                                       Negative binomial                             +0.77   
beta-NLL, beta = 0.25                  Gaussian gene-homoscedastic                   +0.16   
                                       Gaussian heteroscedastic                      +0.65   
beta-NLL, beta = 0.5                   Gaussian gene-homoscedastic                   +0.26   
                                       Gaussian heteroscedastic                      +0.66   
beta-NLL, beta = 0.75                  Gaussian gene-homoscedastic                   +0.31   
                                       Gaussian heteroscedastic                      +0.65   
beta-NLL, beta = 1                     Gaussian gene-homoscedastic                   +0.27   
                                       Gaussian heteroscedastic                      +0.65   
Count heads trained on raw counts      Poisson (raw counts)                          +0.81   
                                       Negative binomial (raw counts)                +0.81   
Uncertainty = coefficient of variation Gaussian gene-homoscedastic                   -0.60   
                                       Gaussian heteroscedastic                      -0.60   
                                       Gaussian spot-homoscedastic                   -0.61   
                                       Poisson                                       -0.79   
                                       Negative binomial                             -0.77   
Uncertainty = 1 + alpha                Negative binomial                             -0.69   

                                                                                     \
                                                                         controlled   
Variation                              Head                                           
Reference (fig:axes)                   Gaussian gene-homoscedastic     -0.02 ± 0.06   
                                       Gaussian heteros

## 4. The resolution of the measurement

For `app:metrics`:*"state the donor-interval half-width for the controlled correlations, expected around 0.02 to 0.03"*.

The half-width is `(hi - lo) / 2` of the Student-t 95% interval over the K held-out donors, taken on the **controlled** correlations only, one value per (variation, cohort, head, axis). Nothing is recomputed here, it is read off `RES`, so it is exactly the `±` printed in the tables above.

Two populations are reported and they answer different questions: the **reference** rows alone, which is what that paragraph describes, and **all variations**, which bounds what the robustness tables show. The last block prints a sentence with the numbers filled in.

In [11]:
# THE RESOLUTION OF THE MEASUREMENT: the half-width of the Student CI over the donors, for the
# CONTROLLED correlations only (it is what the app:metrics sentence calls the resolution:
# below it, a difference is not interpretable).
#
# Nothing is recomputed: the half-width is (hi - lo) / 2 on the already-built RES, hence
# exactly the number that goes into the tables. Two populations are shown, but they do not
# play the same role: the paper's sentence speaks ONLY of the REFERENCE (the fig:axes values),
# and the "all variations" block is there only as reading context for the notebook.
HW = RES[np.isfinite(RES["partial"]) & np.isfinite(RES["partial_lo"])].copy()
HW["half_width"] = (HW["partial_hi"] - HW["partial_lo"]) / 2.0


def hw_table(sub, by):
    g = sub.groupby(by)["half_width"]
    out = pd.DataFrame({"n": g.size(), "min": g.min(), "q25": g.quantile(0.25),
                        "median": g.median(), "q75": g.quantile(0.75), "max": g.max()})
    return out.round(4)


print("=" * 110)
print("RESOLUTION OF THE MEASUREMENT -- half-width of the donor Student-t 95% interval,")
print("CONTROLLED correlations only. One value per (variation, cohort, head, axis).")
print("=" * 110)

for name, sub in (("REFERENCE only (the values of fig:axes) -- THE population the paper quotes",
                   HW[HW["variation"] == "ref"]),
                  ("ALL variations (context for reading the notebook, NOT quoted in the text)",
                   HW)):
    print()
    print(f"--- {name}   [{len(sub)} values]")
    print()
    print("  by axis:")
    print(hw_table(sub, "axis").to_string())
    print()
    print("  by axis x cohort:")
    print(hw_table(sub, ["axis", "dataset"]).to_string())

# Quotes ONLY the reference, on the two axes separately: a single
# value would mix two resolutions that differ by a factor of 3 to 4, and the reader would
# infer a false threshold on one of the two axes.
ref = HW[HW["variation"] == "ref"]
print()
print("=" * 110)
print("SENTENCE FOR app:metrics -- REFERENCE rows only (check the wording):")
print("=" * 110)
for axis, grouping in (("gene", "across the spots of a gene"),
                       ("spot", "across the genes of a spot")):
    s = ref[ref["axis"] == axis]["half_width"]
    no_tls = ref[(ref["axis"] == axis) & (ref["dataset"] != "TLS")]["half_width"]
    tls = ref[(ref["axis"] == axis) & (ref["dataset"] == "TLS")]["half_width"]
    print(f"  {grouping:<28s} {s.min():.3f} to {s.max():.3f}  (median {s.median():.3f}; "
          f"max {no_tls.max():.3f} outside TLS, {tls.max():.3f} on TLS)")

RESOLUTION OF THE MEASUREMENT -- half-width of the donor Student-t 95% interval,
CONTROLLED correlations only. One value per (variation, cohort, head, axis).

--- REFERENCE only (the values of fig:axes) -- THE population the paper quotes   [18 values]

  by axis:
      n     min     q25  median     q75     max
axis                                           
gene  9  0.0054  0.0100  0.0121  0.0149  0.0310
spot  9  0.0078  0.0217  0.0429  0.0550  0.0767

  by axis x cohort:
                  n     min     q25  median     q75     max
axis dataset                                               
gene 10x_TuPro    3  0.0121  0.0135  0.0149  0.0166  0.0183
     LUNG_CANCER  3  0.0054  0.0064  0.0075  0.0087  0.0100
     TLS          3  0.0113  0.0123  0.0133  0.0222  0.0310
spot 10x_TuPro    3  0.0217  0.0269  0.0322  0.0436  0.0550
     LUNG_CANCER  3  0.0078  0.0121  0.0165  0.0297  0.0429
     TLS          3  0.0444  0.0500  0.0556  0.0662  0.0767

--- ALL variations (context for reading th

## 5. Does any variation beat the reference?

The claim `sec:ablations` makes is that no variation ranks the errors better than the reference setting. This tests it, on **both** groupings.

The comparison is **paired**, one pair per (axis, cohort, head): a variation is compared to the reference on the same cohort and the same head, never as an average against an average, since the heads do not share a correlation level and an average would mix the effect of the variation with the composition of the block. The two raw-count heads pair with their depth-corrected twins.

**Better** means a *higher signed* controlled correlation, not a larger absolute one: a negative controlled correlation says the uncertainty points at the *least* wrong predictions, which is the opposite of useful. The threshold is the resolution of section 4, taken as the larger of the pair's two donor-interval half-widths, because `app:metrics` declines to interpret a difference smaller than that interval.

In [12]:
# "Does a variation do BETTER than the reference?" -- posed as a PAIRED comparison, one pair
# per (axis, cohort, head), never as a variation mean against a reference mean: the heads do
# not have the same correlation, so a mean would mix the variation's effect with the block's
# composition.
#
# BETTER = a larger SIGNED controlled correlation. Not |rho|: a NEGATIVE controlled correlation
# means the uncertainty points at the LEAST wrong predictions, which is the opposite of what
# we are after, and counting it as progress through its absolute value would be wrong.
#
# The THRESHOLD is the section 4 resolution, taken as the max of the pair's two half-widths:
# the app:metrics sentence refuses to interpret a difference smaller than the interval, so
# "doing better" can only mean "by more than that".
REF_HEAD = {   # the reference head each variation head compares against
    "poisson_raw":                          "poisson_cp",
    "negativebinomial_hetero_noanchor_raw": "negativebinomial_hetero_noanchor_cp",
}


def _hw(r):
    return (r["partial_hi"] - r["partial_lo"]) / 2.0


pairs = []
for axis in AXES:
    for v in VARIATIONS:
        if v["key"] == "ref":
            continue
        for ds in DATASETS:
            for m in v["models"]:
                r = get(v["key"], ds, m, axis)
                r0 = get("ref", ds, REF_HEAD.get(m, m), axis)
                if r is None or r0 is None:
                    continue
                if not (np.isfinite(r.get("partial", np.nan))
                        and np.isfinite(r0.get("partial", np.nan))):
                    continue
                delta = float(r["partial"] - r0["partial"])
                res = float(max(_hw(r), _hw(r0)))
                pairs.append(dict(axis=axis, variation=v["label"], dataset=ds,
                                  head=MODEL_LABELS.get(m, m), reference=float(r0["partial"]),
                                  variant=float(r["partial"]), delta=delta, resolution=res,
                                  better=bool(delta > res), worse=bool(delta < -res)))
CMP = pd.DataFrame(pairs)

print("=" * 118)
print("IS ANY VARIATION BETTER THAN THE REFERENCE?  Paired per (axis, cohort, head).")
print("better = controlled correlation higher than the reference by MORE than the resolution")
print("(resolution = the larger of the two donor-interval half-widths).")
print("=" * 118)
print(f"{len(CMP)} comparable pairs.")
print()
print(CMP.groupby(["axis", "variation"]).agg(
    pairs=("delta", "size"), better=("better", "sum"), worse=("worse", "sum"),
    delta_min=("delta", "min"), delta_max=("delta", "max")).round(3).to_string())

print()
print("-" * 118)
print("The 10 largest gains over the reference, whatever their size:")
print("-" * 118)
print(CMP.sort_values("delta", ascending=False).head(10)[
    ["axis", "variation", "dataset", "head", "reference", "variant", "delta", "resolution",
     "better"]].round(3).to_string(index=False))

print()
print("=" * 118)
n_better = int(CMP["better"].sum())
if n_better == 0:
    print("VERDICT: no variation exceeds the reference by more than the resolution, on EITHER axis.")
    print(f"         Largest gain anywhere: {CMP['delta'].max():+.3f}, against a resolution of "
          f"{CMP.loc[CMP['delta'].idxmax(), 'resolution']:.3f} on that pair.")
else:
    print(f"VERDICT: {n_better} pair(s) exceed the reference by more than the resolution -- listed "
          f"below. Read them before writing 'no variation does better'.")
    print(CMP[CMP["better"]][["axis", "variation", "dataset", "head", "reference", "variant",
                              "delta", "resolution"]].round(3).to_string(index=False))
print("=" * 118)

IS ANY VARIATION BETTER THAN THE REFERENCE?  Paired per (axis, cohort, head).
better = controlled correlation higher than the reference by MORE than the resolution
(resolution = the larger of the two donor-interval half-widths).
150 comparable pairs.

                                             pairs  better  worse  delta_min  delta_max
axis variation                                                                         
gene Absolute log-ratio error                    9       4      0      0.028      0.211
     Count heads trained on raw counts           3       0      0     -0.005      0.028
     Detected entries only                       9       6      0      0.046      0.241
     Epoch selection: MSE                        9       0      0     -0.007      0.020
     Epoch selection: validation NLL             9       0      1     -0.013      0.029
     Uncertainty = 1 + alpha                     3       0      0     -0.003      0.001
     Uncertainty = coefficient of variation 